# Inspection 16 — Relationship Normalization

## Goal

Normalize and validate Pilot 2 GraphRAG relationships before adding them to the controlled Knowledge Graph.

For each selected raw relationship we will check:

- canonical source and target entities
- allowed predicate
- direction
- modality
- supporting text-unit evidence
- duplicate relationships after entity canonicalization
- final decision: ACCEPTED, CORRECTED, or REJECTED

Raw GraphRAG relationships are treated as extraction candidates, not automatically as authoritative KG facts.

In [1]:
import pandas as pd
from pathlib import Path

relationships_path = "../output/relationships.parquet"
text_units_path = "../output/text_units.parquet"

controlled_kg_path = Path("../../kg/controlled_kg")

relationships = pd.read_parquet(relationships_path)
text_units = pd.read_parquet(text_units_path)

print("Relationships:", len(relationships))
print("Text units:", len(text_units))
print("Relationship columns:", relationships.columns.tolist())

Relationships: 643
Text units: 17
Relationship columns: ['id', 'human_readable_id', 'source', 'target', 'description', 'weight', 'combined_degree', 'text_unit_ids']


## Starting point

Entity canonicalization was completed first in `inspection_15_entity_normalization.ipynb`.

The current controlled KG prototype contains canonical entities such as:

- `TEC_0001` — Photovoltaikanlage
- `TEC_0002` — Agri-PV-Anlage
- `ORG_0001` — BMK / full ministry name
- `POL_0001` — Österreichische Photovoltaik-Strategie

Relationship normalization uses these canonical entity IDs rather than the raw GraphRAG names.

One normalized relationship has already been identified:

`ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

A second raw relationship was inspected and rejected:

`AGRI-PV-ANLAGE -- PART_OF --> FOTOVOLTAIKANLAGE`

because its supporting source text did not justify the `PART_OF` predicate.

In [2]:
canonical_entities_path = controlled_kg_path / "canonical_entities_working.csv"
normalized_relationships_path = controlled_kg_path / "normalized_relationships_working.csv"

canonical_entities = pd.read_csv(canonical_entities_path)
normalized_relationships = pd.read_csv(normalized_relationships_path)

print("Canonical entities:", len(canonical_entities))
print("Normalized relationships:", len(normalized_relationships))

display(canonical_entities)
display(normalized_relationships)

Canonical entities: 4
Normalized relationships: 1


,canonical_id,canonical_label,entity_type,aliases,raw_entity_ids,description,validation_status,notes
0,TEC_0001,Photovoltaikanlage,TECHNOLOGY,PV-ANLAGE; PV-ANLAGEN; PHOTOVOLTAIKANLAGEN; FO...,3b926070-fa35-4f04-ba30-185a40c5d707; ade05329...,Photovoltaic installation or system used to ge...,CORRECTED,Merged four raw GraphRAG nodes representing si...
1,TEC_0002,Agri-PV-Anlage,TECHNOLOGY,AGRI-PV-ANLAGE; AGRI-PV-ANLAGEN,077391bf-ba15-4ade-9a43-449741d7f5be; 10850660...,Photovoltaic installation enabling combined ag...,CORRECTED,Merged singular and plural GraphRAG nodes refe...
2,ORG_0001,"Bundesministerium für Klimaschutz, Umwelt, Ene...",ORGANIZATION,"BMK; BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT...",86fa5f94-0db2-4a5f-8d24-d32bb067610b; 70ab370d...,Austrian federal ministry responsible for clim...,CORRECTED,Merged acronym BMK and two full-name GraphRAG ...
3,POL_0001,Österreichische Photovoltaik-Strategie,POLICY,PHOTOVOLTAIK-STRATEGIE; ÖSTERREICHISCHE PHOTOV...,fac405b8-06d4-43da-b1fc-6e707fd0cade; 7104f7f4...,Austrian national strategy providing a framewo...,CORRECTED,Merged short and full GraphRAG names referring...


,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0001,ORG_0001,RESPONSIBLE_FOR,POL_0001,EXPLICIT_FACT,NaN,766acd95-2d5a-4286-bf19-3455bb2616ec; 0d51bb6b...,d7219f76b31818b7b13bf08dda81003d752e53f7660831...,CORRECTED,Collapsed two duplicate GraphRAG relationships...


## Case 1 — Rejected relationship

Raw GraphRAG relationship 257 connects:

`AGRI-PV-ANLAGE -- PART_OF --> FOTOVOLTAIKANLAGE`

This case is used to test whether the extracted predicate is actually supported by the source evidence.

In [3]:
relation_257 = relationships[
    relationships["human_readable_id"] == 257
]

relation_257[
    [
        "id",
        "human_readable_id",
        "source",
        "target",
        "description",
        "weight",
        "text_unit_ids"
    ]
]

,id,human_readable_id,source,target,description,weight,text_unit_ids
257,1b0a059f-694c-4e24-a3c7-766ebedc83a4,257,AGRI-PV-ANLAGE,FOTOVOLTAIKANLAGE,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...,10.0,[23663988970639bea940978ccbd9150fd292856f2f31a...


In [4]:
supporting_text_unit_id = relation_257.iloc[0]["text_unit_ids"][0]

evidence_257 = text_units[
    text_units["id"] == supporting_text_unit_id
]

print("TEXT UNIT ID:")
print(supporting_text_unit_id)

print("\nSOURCE TEXT:\n")
print(evidence_257.iloc[0]["text"])

TEXT UNIT ID:
23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24

SOURCE TEXT:

 bevorzugte PV-An-

wendungen, sowie eine Aufteilung der Fördermittel in Größenklassen sind darin bereits ent-
halten.  Auch  die  Novellierung  des  Wohnungseigentumsgesetzes  im  Jänner  20227 hat  Er-
leichterungen für den Bau von Photovoltaikanlagen bei Reihenhäusern oder Einzelgebäu-

den gebracht. Das Anfang 2024 in Begutachtung befindliche Elektrizitätswirtschaftsgesetz

(ElWG) schafft neue und zeitgemäße Spielregeln für den Strommarkt. Durch mehr Transpa-

renz im Netz, neue Marktrollen und Maßnahmen für mehr Flexibilität wird ein wichtiger

Beitrag zur schnelleren Integration von erneuerbaren Energieanlagen geschaffen. Weiters

wird auch das in Vorbereitung stehende Erneuerbaren-Ausbau-Beschleunigungsgesetz (E-

ABG)  zu  einer  Verfahrensbeschleunigung  bei  der  Errichtung  von  PV-Anlagen  und  für  die

Energiewende n

### Evaluation

The supporting source text contains the statement:

> "Im Bereich der Landwirtschaft sollen Anlagen in Form von Agri-PV-Anlagen konzipiert werden."

This supports Agri-PV as a specific form/application of photovoltaic installation
in the agricultural context.

However, the source does **not** state that an Agri-PV installation is physically
`PART_OF` another photovoltaic installation.

Therefore the GraphRAG relationship:

`AGRI-PV-ANLAGE -- PART_OF --> FOTOVOLTAIKANLAGE`

uses a misleading predicate and is not accepted into the controlled Knowledge Graph.

**Decision: REJECTED**

A relation such as `SUBTYPE_OF` or `IS_A` might conceptually represent the meaning
better, but these predicates are not currently defined in Schema V1.1.

The schema is therefore not extended based on this single example.

In [5]:
relation_257_decision = pd.DataFrame([
    {
        "raw_relationship_id": relation_257.iloc[0]["id"],
        "human_readable_id": 257,
        "raw_source": "AGRI-PV-ANLAGE",
        "raw_predicate": "PART_OF",
        "raw_target": "FOTOVOLTAIKANLAGE",
        "canonical_source_id": "TEC_0002",
        "canonical_target_id": "TEC_0001",
        "decision": "REJECTED",
        "reason": (
            "The source discusses Agri-PV as a form/application of PV in agriculture, "
            "but does not support the predicate PART_OF. The extracted relationship "
            "therefore overstates the source evidence."
        ),
        "text_unit_id": supporting_text_unit_id
    }
])

relation_257_decision

,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,1b0a059f-694c-4e24-a3c7-766ebedc83a4,257,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,TEC_0002,TEC_0001,REJECTED,The source discusses Agri-PV as a form/applica...,23663988970639bea940978ccbd9150fd292856f2f31a1...


In [6]:
relationship_decisions_path = (
    controlled_kg_path / "relationship_decisions_working.csv"
)

relation_257_decision.to_csv(
    relationship_decisions_path,
    index=False
)

print("Saved:", relationship_decisions_path)

Saved: ..\..\kg\controlled_kg\relationship_decisions_working.csv


In [7]:
relationship_decisions = pd.read_csv(
    relationship_decisions_path
)

relationship_decisions

,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,1b0a059f-694c-4e24-a3c7-766ebedc83a4,257,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,TEC_0002,TEC_0001,REJECTED,The source discusses Agri-PV as a form/applica...,23663988970639bea940978ccbd9150fd292856f2f31a1...


## Case 2 — Accepted / normalized relationship

GraphRAG extracted two relationships connecting the Austrian ministry (BMK)
to the Austrian Photovoltaic Strategy.

After entity canonicalization, both raw relationships refer to the same endpoints:

`ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

This case tests duplicate collapsing and modality selection.

In [8]:
case2 = relationships[
    relationships["human_readable_id"].isin([0, 642])
]

case2[
    [
        "id",
        "human_readable_id",
        "source",
        "target",
        "description",
        "weight",
        "text_unit_ids"
    ]
]

,id,human_readable_id,source,target,description,weight,text_unit_ids
0,766acd95-2d5a-4286-bf19-3455bb2616ec,0,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...,10.0,[d7219f76b31818b7b13bf08dda81003d752e53f766083...
642,0d51bb6b-e917-4703-b77d-aef2fc025f29,642,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=REAS...,6.0,[7ae981575e185e2452d144d541d822b7725e27bc7dda5...


In [9]:
for _, row in case2.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("HUMAN READABLE ID:", row["human_readable_id"])
    print("GRAPH RELATIONSHIP ID:", row["id"])
    print("SOURCE:")
    print(row["source"])
    print("TARGET:")
    print(row["target"])
    print("DESCRIPTION:")
    print(row["description"])
    print("WEIGHT:", row["weight"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

HUMAN READABLE ID: 0
GRAPH RELATIONSHIP ID: 766acd95-2d5a-4286-bf19-3455bb2616ec
SOURCE:
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)
TARGET:
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
DESCRIPTION:
[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPLICIT_FACT] The BMK is listed as the publisher and issuer of the Austrian Photovoltaic Strategy.
WEIGHT: 10.0
TEXT UNIT IDS:
['d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475']

HUMAN READABLE ID: 642
GRAPH RELATIONSHIP ID: 0d51bb6b-e917-4703-b77d-aef2fc025f29
SOURCE:
BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE
TARGET:
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
DESCRIPTION:
[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=REASONABLE_INFERENCE] The Federal Ministry for Climate Action, Environment, Energy, Mobility, Innovation and Technology is the government body responsible for the Austr

In [10]:
case2_text_unit_ids = []

for _, row in case2.iterrows():
    case2_text_unit_ids.extend(list(row["text_unit_ids"]))

case2_evidence = text_units[
    text_units["id"].isin(case2_text_unit_ids)
]

for _, row in case2_evidence.iterrows():
    print("=" * 100)
    print("TEXT UNIT ID:")
    print(row["id"])
    print("\nSOURCE TEXT:\n")
    print(row["text"])
    print()

TEXT UNIT ID:
d7219f76b31818b7b13bf08dda81003d752e53f7660831d5e201d105d2bc1fc19e8cc49ca6cd21e08570939d7fb3e87b45f0f5feb8575df195e7be8cbd3b6475

SOURCE TEXT:

Österreichische Photovoltaik-
Strategie

Zielsetzungen und Aktionsfelder eines strategischen Ausbauprozesses

sowie Maßnahmen für einen koordinierten Ausbau der Photovoltaik

in Österreich

Impressum

Medieninhaber, Verleger und Herausgeber:

Bundesministerium für Klimaschutz, Umwelt, Energie, Mobilität,

Innovation und Technologie, Radetzkystraße 2, 1030 Wien

Autor: Hubert Fechner

Fotonachweis: stock.adobe.com – Alan (Cover), BMK/Cajetan Perwein (Vorwort)

Wien, 2024.

Vorwort

Die Klimakrise ist eine der größten Herausforderung unserer Zeit.

Wir spüren ihre Auswirkungen immer deutlicher. Als österreichi-

sche  Bundesregierung haben  wir uns  daher  ein ehrgeiziges  Ziel

gesetzt: ein klimaneutrales Österreich bis 2040. Zur Bekämpfung

der Klimakrise sind viele Maßnahmen notwendig, insbesondere

ein Vorantreiben der Energie

In [11]:
relation_642 = relationships[
    relationships["human_readable_id"] == 642
]

text_unit_642_id = relation_642.iloc[0]["text_unit_ids"][0]

evidence_642 = text_units[
    text_units["id"] == text_unit_642_id
]

print("TEXT UNIT ID:")
print(text_unit_642_id)

print("\nSOURCE TEXT:\n")
print(evidence_642.iloc[0]["text"])

TEXT UNIT ID:
7ae981575e185e2452d144d541d822b7725e27bc7dda501c9d8495719003ccb89584fe5e1ac83baa67ae4060497ae5e2536fae015a551dc8cdefa9b5aa17289b

SOURCE TEXT:

att

Terawattstunde

Umweltverträglichkeitsprüfung

Österreichische Photovoltaik-Strategie

41 von 44

42 von 44

Österreichische Photovoltaik-Strategie

Österreichische Photovoltaik-Strategie

43 von 44

Bundesministerium für Klimaschutz, Umwelt, Energie, Mobilität,

Innovation und Technologie

Radetzkystraße 2, 1030 Wien

+43 (0) 800 21 53 59

servicebuero@bmk.gv.at

bmk.gv.at




### Evaluation

The two raw GraphRAG relationships resolve to the same canonical endpoints:

`ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

However, their evidence quality differs.

Relationship `0` is supported by the document imprint, which explicitly identifies
the BMK as:

> "Medieninhaber, Verleger und Herausgeber"

of the Austrian Photovoltaic Strategy.

Relationship `642` is based only on the ministry being listed in the strategy
document's closing/contact information. This supports an association with the
document but does not independently establish responsibility.

The two raw relationships are therefore treated as duplicate extractions after
entity canonicalization.

Because Schema V1.1 does not contain a dedicated `PUBLISHER_OF` relation,
the explicit publisher/issuer role is represented using the controlled predicate
`RESPONSIBLE_FOR`.

The stronger explicit evidence determines the normalized modality.

**Normalized relationship**

`REL_0001: ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

- Modality: `EXPLICIT_FACT`
- Validation status: `CORRECTED`
- Raw relationships collapsed: `0`, `642`
- Both raw relationship IDs and text-unit IDs are preserved as provenance.

The GraphRAG relationship weights (10.0 and 6.0) are not interpreted as
truth probabilities.

## Case 3 — Candidate for correction

This case looks for a GraphRAG relationship where the underlying claim may be
supported by the source, but the predicate, direction, modality, or endpoint
representation may require correction before entering the controlled KG.

In [12]:
agri_candidates = relationships[
    relationships["human_readable_id"].isin([136, 292, 425])
]

agri_candidates[
    [
        "id",
        "human_readable_id",
        "source",
        "target",
        "description",
        "weight",
        "text_unit_ids"
    ]
]

,id,human_readable_id,source,target,description,weight,text_unit_ids
136,e1e2e142-2dd0-40b8-95ae-57cfde04808a,136,AGRI-PV-ANLAGE,LANDWIRT:INNEN,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[e498566a78d36b3ce13304157993b9b4269045840f1d6...
292,4ae7c604-c0df-481b-8f30-4dc6e4cd355b,292,AGRI-PV-ANLAGEN,LANDWIRTSCHAFT,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[41bd25afdc1cce74b36dd6db5e44e91354172fda16475...
425,94f1ba26-130a-4a14-9023-f55ecf185196,425,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,AGRI-PV-ANLAGE,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...,9.0,[4d221520fe7090409cfa9cda598db30683356c340ab35...


In [13]:
relation_425 = relationships[
    relationships["human_readable_id"] == 425
]

for _, row in relation_425.iterrows():
    print("GRAPH RELATIONSHIP ID:", row["id"])
    print("SOURCE:", row["source"])
    print("TARGET:", row["target"])
    print("DESCRIPTION:")
    print(row["description"])
    print("WEIGHT:", row["weight"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])

GRAPH RELATIONSHIP ID: 94f1ba26-130a-4a14-9023-f55ecf185196
SOURCE: ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
TARGET: AGRI-PV-ANLAGE
DESCRIPTION:
[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FACT] The strategy encourages agricultural dual-use through Agri-PV systems to increase acceptance and multifunctionality.
WEIGHT: 9.0
TEXT UNIT IDS:
['4d221520fe7090409cfa9cda598db30683356c340ab3584a063ffe92486f2fe7225fc056f8b997f085ff0c0f7475828879fa678b7b1709148cb03584b292dab0']


In [14]:
text_unit_425_id = relation_425.iloc[0]["text_unit_ids"][0]

evidence_425 = text_units[
    text_units["id"] == text_unit_425_id
]

print("TEXT UNIT ID:")
print(text_unit_425_id)

print("\nSOURCE TEXT:\n")
print(evidence_425.iloc[0]["text"])

TEXT UNIT ID:
4d221520fe7090409cfa9cda598db30683356c340ab3584a063ffe92486f2fe7225fc056f8b997f085ff0c0f7475828879fa678b7b1709148cb03584b292dab0

SOURCE TEXT:

, um zu klären, ob und in

welcher Weise eine Ausweitung auf größere Speichersysteme und größere Anlagen wirk-

sam dazu beiträgt, die in dieser PV-Strategie skizzierten Ziele zu erreichen.

26 von 44

Österreichische Photovoltaik-Strategie

6.4  Aktionsfeld „Akzeptanz“

Mit dem Ort der Realisierung direkt verbunden ist die Erhaltung der grundsätzlich hohen

Akzeptanz des PV-Ausbaus. Dies wird dann gelingen, wenn

•  neben dem individuellen Nutzen auch ein Nutzen für die Bevölkerung sichergestellt ist

(vor allem bei größeren Projekten),

•  ästhetische Fragen im Bauwesen adressiert sind, wobei Bautradition mit dem Wagnis

zu Neuem maßvoll verbunden wird,

•  dem Naturschutz in gebotener Weise entsprochen wird, Freiflächenanlagen wenn

möglich als Biodiversitätsanlagen15 errichtet werden,

•  die Qualität bei Planung und Umsetzun

### Evaluation

The supporting source text states:

> "Hier gilt es weiterhin die landwirtschaftliche Doppelnutzung im Rahmen von Agri-PV-Anlagen zu fördern."

This explicitly recommends continued promotion of agricultural dual use through
Agri-PV installations.

The GraphRAG predicate `SUPPORTS` is therefore a reasonable controlled
representation of the relationship between the Austrian Photovoltaic Strategy
and Agri-PV.

However, the extracted modality `EXPLICIT_FACT` does not capture the normative
meaning of the source statement. The wording "gilt es ... zu fördern" expresses
a recommendation for action rather than a neutral factual assertion.

**Decision: CORRECTED**

Normalized relationship:

`POL_0001 -- SUPPORTS --> TEC_0002`

- Predicate: `SUPPORTS`
- Modality: `RECOMMENDATION`
- Validation status: `CORRECTED`

The relationship is retained in the controlled KG, but its modality is corrected
from `EXPLICIT_FACT` to `RECOMMENDATION`.

In [15]:
second_normalized_relationship = pd.DataFrame([
    {
        "relationship_id": "REL_0002",
        "source_id": "POL_0001",
        "predicate": "SUPPORTS",
        "target_id": "TEC_0002",
        "modality": "RECOMMENDATION",
        "status": "",
        "raw_relationship_ids": relation_425.iloc[0]["id"],
        "text_unit_ids": text_unit_425_id,
        "validation_status": "CORRECTED",
        "normalization_note": (
            "The source explicitly recommends continued promotion of agricultural "
            "dual use through Agri-PV. SUPPORTS is retained, but GraphRAG modality "
            "EXPLICIT_FACT is corrected to RECOMMENDATION."
        )
    }
])

second_normalized_relationship

,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0002,POL_0001,SUPPORTS,TEC_0002,RECOMMENDATION,,94f1ba26-130a-4a14-9023-f55ecf185196,4d221520fe7090409cfa9cda598db30683356c340ab358...,CORRECTED,The source explicitly recommends continued pro...


In [16]:
normalized_relationships = pd.read_csv(normalized_relationships_path)

if "REL_0002" not in normalized_relationships["relationship_id"].values:
    normalized_relationships = pd.concat(
        [normalized_relationships, second_normalized_relationship],
        ignore_index=True
    )

    normalized_relationships.to_csv(
        normalized_relationships_path,
        index=False
    )

    print("REL_0002 saved successfully.")
else:
    print("REL_0002 already exists.")

normalized_relationships

REL_0002 saved successfully.


,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0001,ORG_0001,RESPONSIBLE_FOR,POL_0001,EXPLICIT_FACT,NaN,766acd95-2d5a-4286-bf19-3455bb2616ec; 0d51bb6b...,d7219f76b31818b7b13bf08dda81003d752e53f7660831...,CORRECTED,Collapsed two duplicate GraphRAG relationships...
1,REL_0002,POL_0001,SUPPORTS,TEC_0002,RECOMMENDATION,,94f1ba26-130a-4a14-9023-f55ecf185196,4d221520fe7090409cfa9cda598db30683356c340ab358...,CORRECTED,The source explicitly recommends continued pro...


In [17]:
relation_425_decision = pd.DataFrame([
    {
        "raw_relationship_id": relation_425.iloc[0]["id"],
        "human_readable_id": 425,
        "raw_source": "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE",
        "raw_predicate": "SUPPORTS",
        "raw_target": "AGRI-PV-ANLAGE",
        "canonical_source_id": "POL_0001",
        "canonical_target_id": "TEC_0002",
        "decision": "CORRECTED",
        "reason": (
            "The predicate SUPPORTS is supported by the source, but the raw modality "
            "EXPLICIT_FACT was incorrect. The source recommends continued promotion "
            "of Agri-PV, so the normalized modality is RECOMMENDATION."
        ),
        "text_unit_id": text_unit_425_id
    }
])

relation_425_decision

,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,94f1ba26-130a-4a14-9023-f55ecf185196,425,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,AGRI-PV-ANLAGE,POL_0001,TEC_0002,CORRECTED,The predicate SUPPORTS is supported by the sou...,4d221520fe7090409cfa9cda598db30683356c340ab358...


In [18]:
relationship_decisions = pd.read_csv(relationship_decisions_path)

if 425 not in relationship_decisions["human_readable_id"].values:
    relationship_decisions = pd.concat(
        [relationship_decisions, relation_425_decision],
        ignore_index=True
    )

    relationship_decisions.to_csv(
        relationship_decisions_path,
        index=False
    )

    print("Relationship 425 decision saved.")
else:
    print("Relationship 425 decision already exists.")

relationship_decisions

Relationship 425 decision saved.


,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,1b0a059f-694c-4e24-a3c7-766ebedc83a4,257,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,TEC_0002,TEC_0001,REJECTED,The source discusses Agri-PV as a form/applica...,23663988970639bea940978ccbd9150fd292856f2f31a1...
1,94f1ba26-130a-4a14-9023-f55ecf185196,425,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,AGRI-PV-ANLAGE,POL_0001,TEC_0002,CORRECTED,The predicate SUPPORTS is supported by the sou...,4d221520fe7090409cfa9cda598db30683356c340ab358...


## Automatic validation preparation

The previous cases were manually inspected to identify representative
relationship-normalization problems.

The next stage applies systematic checks across all raw Pilot 2 relationships.

The first step extracts the structured `RELATION_TYPE` and `MODALITY`
markers from the GraphRAG relationship descriptions so that the complete
relationship set can be analyzed automatically.

This does not automatically accept or reject relationships.
It prepares the raw relationship table for validation.

In [19]:
import re

def extract_marker(text, marker):
    if pd.isna(text):
        return None

    match = re.search(
        rf"\[{marker}=([^\]]+)\]",
        str(text)
    )

    return match.group(1).strip() if match else None


relationships_checked = relationships.copy()

relationships_checked["relation_type"] = (
    relationships_checked["description"]
    .apply(lambda x: extract_marker(x, "RELATION_TYPE"))
)

relationships_checked["modality"] = (
    relationships_checked["description"]
    .apply(lambda x: extract_marker(x, "MODALITY"))
)

print("Relationships:", len(relationships_checked))

print("\nRelation types:")
print(
    relationships_checked["relation_type"]
    .value_counts(dropna=False)
)

print("\nModalities:")
print(
    relationships_checked["modality"]
    .value_counts(dropna=False)
)

Relationships: 643

Relation types:
relation_type
SUPPORTS           164
ASSOCIATED_WITH    107
CONTRIBUTES_TO      70
PROPOSES            44
PART_OF             39
MEASURED_BY         37
CONSTRAINS          33
RESPONSIBLE_FOR     25
SETS_TARGET         18
HAS_TARGET          18
REGULATES           15
IMPLEMENTS          11
SUPPORTED_BY        11
REQUIRES            10
None                 9
LOCATED_IN           9
ALIAS_OF             7
FUNDS                6
APPLIES_TO           4
IMPLEMENTED_BY       2
BENEFITS_FROM        1
IMPLEMENTED          1
CAN_USE              1
AMENDS               1
Name: count, dtype: int64

Modalities:
modality
EXPLICIT_FACT           550
PLANNED_ACTION           26
RECOMMENDATION           19
SCENARIO                 18
PROPOSAL                 13
None                      9
LEGAL_REQUIREMENT         5
REASONABLE_INFERENCE      3
Name: count, dtype: int64


## Automatic schema-compliance checks

The complete Pilot 2 relationship set is now checked against the controlled
relationship vocabulary and modality vocabulary defined in Schema V1.1.

These checks do not determine whether a relationship is factually correct.

They identify structural problems such as:

- out-of-schema predicates
- missing predicates
- out-of-schema modalities
- missing modalities

Relationships that fail these checks are flagged for later normalization or review.

In [20]:
allowed_relation_types = {
    "HAS_TARGET",
    "SETS_TARGET",
    "CONTRIBUTES_TO",
    "SUPPORTS",
    "CONSTRAINS",
    "IMPLEMENTS",
    "PROPOSES",
    "REQUIRES",
    "RESPONSIBLE_FOR",
    "APPLIES_TO",
    "REGULATES",
    "FUNDS",
    "MEASURED_BY",
    "HAS_VALUE",
    "PART_OF",
    "LOCATED_IN",
    "ALIAS_OF"
}

allowed_modalities = {
    "EXPLICIT_FACT",
    "LEGAL_REQUIREMENT",
    "RECOMMENDATION",
    "PROPOSAL",
    "PLANNED_ACTION",
    "SCENARIO",
    "REASONABLE_INFERENCE"
}

relationships_checked["predicate_valid"] = (
    relationships_checked["relation_type"]
    .isin(allowed_relation_types)
)

relationships_checked["modality_valid"] = (
    relationships_checked["modality"]
    .isin(allowed_modalities)
)

print("Total relationships:", len(relationships_checked))

print(
    "Valid predicates:",
    relationships_checked["predicate_valid"].sum()
)

print(
    "Invalid / missing predicates:",
    (~relationships_checked["predicate_valid"]).sum()
)

print(
    "Valid modalities:",
    relationships_checked["modality_valid"].sum()
)

print(
    "Invalid / missing modalities:",
    (~relationships_checked["modality_valid"]).sum()
)

print("\nOut-of-schema / missing predicates:")

print(
    relationships_checked.loc[
        ~relationships_checked["predicate_valid"],
        "relation_type"
    ].value_counts(dropna=False)
)

Total relationships: 643
Valid predicates: 510
Invalid / missing predicates: 133
Valid modalities: 634
Invalid / missing modalities: 9

Out-of-schema / missing predicates:
relation_type
ASSOCIATED_WITH    107
SUPPORTED_BY        11
None                 9
IMPLEMENTED_BY       2
BENEFITS_FROM        1
IMPLEMENTED          1
CAN_USE              1
AMENDS               1
Name: count, dtype: int64


## Structural validation flags

Each raw relationship is assigned a structural validation flag.

This check only evaluates whether the extracted predicate and modality comply
with Schema V1.1.

It does not determine whether the relationship is semantically correct or
supported by the source evidence.

Possible flags:

- `STRUCTURALLY_OK`
- `INVALID_PREDICATE`
- `INVALID_MODALITY`
- `INVALID_PREDICATE_AND_MODALITY`

Relationships flagged as structurally valid may still require semantic
or provenance-based validation.

In [21]:
def structural_flag(row):
    if row["predicate_valid"] and row["modality_valid"]:
        return "STRUCTURALLY_OK"

    elif not row["predicate_valid"] and row["modality_valid"]:
        return "INVALID_PREDICATE"

    elif row["predicate_valid"] and not row["modality_valid"]:
        return "INVALID_MODALITY"

    else:
        return "INVALID_PREDICATE_AND_MODALITY"


relationships_checked["validation_flag"] = (
    relationships_checked.apply(
        structural_flag,
        axis=1
    )
)

print("Structural validation results:\n")

print(
    relationships_checked["validation_flag"]
    .value_counts()
)

Structural validation results:

validation_flag
STRUCTURALLY_OK                   510
INVALID_PREDICATE                 124
INVALID_PREDICATE_AND_MODALITY      9
Name: count, dtype: int64


## Inspection of out-of-schema predicate groups

Relationships with invalid or missing predicates are inspected by predicate group.

Only a small sample from each group is displayed initially. The purpose is to
determine whether a group can:

- be normalized automatically to an existing Schema V1.1 predicate,
- requires a direction change,
- requires semantic/source review, or
- should be rejected/quarantined.

No automatic predicate mapping is performed yet.

In [22]:
invalid_predicate_examples = (
    relationships_checked[
        ~relationships_checked["predicate_valid"]
    ]
    .copy()
)

invalid_predicate_examples["relation_type_display"] = (
    invalid_predicate_examples["relation_type"]
    .fillna("MISSING")
)

sample_invalid = (
    invalid_predicate_examples
    .sort_values(
        ["relation_type_display", "human_readable_id"]
    )
    .groupby("relation_type_display", group_keys=False)
    .head(3)
)

sample_invalid[
    [
        "human_readable_id",
        "source",
        "target",
        "relation_type_display",
        "modality",
        "description",
        "validation_flag"
    ]
]

,human_readable_id,source,target,relation_type_display,modality,description,validation_flag
607,607,RICHTLINIE (EU) 2023/2413 (RED III),RICHTLINIE (EU) 2018/2001,AMENDS,EXPLICIT_FACT,[RELATION_TYPE=AMENDS] [MODALITY=EXPLICIT_FACT...,INVALID_PREDICATE
1,1,LEONORE GEWESSLER,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ASSOCIATED_WITH,EXPLICIT_FACT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,INVALID_PREDICATE
16,16,PHOTOVOLTAIK-ANLAGE,PHOTOVOLTAIK,ASSOCIATED_WITH,EXPLICIT_FACT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,INVALID_PREDICATE
18,18,STROMVERSORGUNG,PHOTOVOLTAIK,ASSOCIATED_WITH,EXPLICIT_FACT,[RELATION_TYPE=ASSOCIATED_WITH] [MODALITY=EXPL...,INVALID_PREDICATE
281,281,LANDWIRTSCHAFT,AGRI-PV-ANLAGE,BENEFITS_FROM,EXPLICIT_FACT,[RELATION_TYPE=BENEFITS_FROM] [MODALITY=EXPLIC...,INVALID_PREDICATE
578,578,AKTIVE KUND:INNEN,GEMEINSCHAFTLICHE ERZEUGUNGSANLAGEN (GEA),CAN_USE,EXPLICIT_FACT,[RELATION_TYPE=CAN_USE] [MODALITY=EXPLICIT_FAC...,INVALID_PREDICATE
404,404,AKTIONSPLAN NETZANSCHLUSS,E-CONTROL,IMPLEMENTED,EXPLICIT_FACT,[RELATION_TYPE=IMPLEMENTED] [MODALITY=EXPLICIT...,INVALID_PREDICATE
253,253,ZUSCHLÄGE BEI INNOVATIVEN ANLAGENFORMEN,ERNEUERBAREN-AUSBAU-GESETZ (EAG),IMPLEMENTED_BY,EXPLICIT_FACT,[RELATION_TYPE=IMPLEMENTED_BY] [MODALITY=EXPLI...,INVALID_PREDICATE
254,254,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,ERNEUERBAREN-AUSBAU-GESETZ (EAG),IMPLEMENTED_BY,EXPLICIT_FACT,[RELATION_TYPE=IMPLEMENTED_BY] [MODALITY=EXPLI...,INVALID_PREDICATE
2,2,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,PHOTOVOLTAIK,MISSING,None,The Austrian Photovoltaic Strategy (ÖSTERREICH...,INVALID_PREDICATE_AND_MODALITY


## Initial treatment of out-of-schema predicates

The invalid predicate groups are not automatically rewritten.

They are first classified into broad normalization categories:

- `POSSIBLE_INVERSE_MAPPING` — predicate may correspond to an allowed predicate
  after reversing source and target.
- `REVIEW_REQUIRED` — semantic meaning must be inspected before mapping.
- `MISSING_STRUCTURE` — GraphRAG did not provide a structured predicate/modality.
- `QUARANTINE` — predicate is outside Schema V1.1 and should not enter the
  controlled KG without explicit review.

Initial hypotheses:

- `SUPPORTED_BY` → possible inverse of `SUPPORTS`
- `IMPLEMENTED_BY` → possible inverse of `IMPLEMENTS`
- `ASSOCIATED_WITH` → review required because it is too vague
- `BENEFITS_FROM` → review required
- `CAN_USE` → review required
- `IMPLEMENTED` → review required
- `AMENDS` → quarantine/review; do not extend the schema based on one case
- missing predicate → missing structure / review

These are hypotheses only. No automatic mapping is applied yet.

In [23]:
predicate_group_actions = {
    "SUPPORTED_BY": "POSSIBLE_INVERSE_MAPPING",
    "IMPLEMENTED_BY": "POSSIBLE_INVERSE_MAPPING",
    "ASSOCIATED_WITH": "REVIEW_REQUIRED",
    "BENEFITS_FROM": "REVIEW_REQUIRED",
    "CAN_USE": "REVIEW_REQUIRED",
    "IMPLEMENTED": "REVIEW_REQUIRED",
    "AMENDS": "QUARANTINE",
    "MISSING": "MISSING_STRUCTURE"
}

invalid_predicate_examples["group_action"] = (
    invalid_predicate_examples["relation_type_display"]
    .map(predicate_group_actions)
)

print(
    invalid_predicate_examples[
        ["relation_type_display", "group_action"]
    ]
    .value_counts()
    .sort_index()
)

relation_type_display  group_action            
AMENDS                 QUARANTINE                    1
ASSOCIATED_WITH        REVIEW_REQUIRED             107
BENEFITS_FROM          REVIEW_REQUIRED               1
CAN_USE                REVIEW_REQUIRED               1
IMPLEMENTED            REVIEW_REQUIRED               1
IMPLEMENTED_BY         POSSIBLE_INVERSE_MAPPING      2
MISSING                MISSING_STRUCTURE             9
SUPPORTED_BY           POSSIBLE_INVERSE_MAPPING     11
Name: count, dtype: int64


## Testing the `SUPPORTED_BY` inverse-mapping hypothesis

`SUPPORTED_BY` is not part of Schema V1.1.

A possible normalization is:

`A -- SUPPORTED_BY --> B`

→

`B -- SUPPORTS --> A`

Before applying this automatically, all Pilot 2 `SUPPORTED_BY` examples are
inspected to determine whether reversing the direction preserves their meaning.

No relationship is modified at this stage.

In [24]:
supported_by_cases = relationships_checked[
    relationships_checked["relation_type"] == "SUPPORTED_BY"
].copy()

print("Number of SUPPORTED_BY relationships:", len(supported_by_cases))

supported_by_cases[
    [
        "human_readable_id",
        "source",
        "target",
        "modality",
        "description",
        "text_unit_ids"
    ]
]

Number of SUPPORTED_BY relationships: 11


,human_readable_id,source,target,modality,description,text_unit_ids
272,272,PV-FREIFLÄCHENANLAGE,ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN,EXPLICIT_FACT,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=EXPLICI...,[23663988970639bea940978ccbd9150fd292856f2f31a...
273,273,VERSIEGELTE FLÄCHEN,FÖRDERUNGEN FÜR PHOTOVOLTAIK,EXPLICIT_FACT,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=EXPLICI...,[23663988970639bea940978ccbd9150fd292856f2f31a...
278,278,GRÜNDACH,KUMULIERTE FÖRDERUNG VON PV UND GRÜNDACH,PROPOSAL,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PROPOSA...,[23663988970639bea940978ccbd9150fd292856f2f31a...
279,279,UNTERKONSTRUKTION,KUMULIERTE FÖRDERUNG VON PV UND GRÜNDACH,PROPOSAL,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PROPOSA...,[23663988970639bea940978ccbd9150fd292856f2f31a...
282,282,NETZANSCHLUSS,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...
283,283,NETZZUGANG,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...
285,285,NETZKAPAZITÄTEN,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...
286,286,NETZENTWICKLUNGSPLÄNE,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...
287,287,AMTLICH BEFASSTE MARKTROLLEN,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...
288,288,FLEXIBILITÄT,ELEKTRIZITÄTSWIRTSCHAFTSGESETZ (ELWG),PLANNED_ACTION,[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PLANNED...,[23663988970639bea940978ccbd9150fd292856f2f31a...


In [25]:
for _, row in supported_by_cases.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("ID:", row["human_readable_id"])
    print("SOURCE:", row["source"])
    print("TARGET:", row["target"])
    print("MODALITY:", row["modality"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

ID: 272
SOURCE: PV-FREIFLÄCHENANLAGE
TARGET: ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN
MODALITY: EXPLICIT_FACT
DESCRIPTION:
[RELATION_TYPE=SUPPORTED_BY] [MODALITY=EXPLICIT_FACT] Open-space PV installations are subject to financial reductions.
TEXT UNIT IDS:
['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24']

ID: 273
SOURCE: VERSIEGELTE FLÄCHEN
TARGET: FÖRDERUNGEN FÜR PHOTOVOLTAIK
MODALITY: EXPLICIT_FACT
DESCRIPTION:
[RELATION_TYPE=SUPPORTED_BY] [MODALITY=EXPLICIT_FACT] Sealed surfaces are to be mobilized for PV with the help of support schemes and obligations.
TEXT UNIT IDS:
['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24']

ID: 278
SOURCE: GRÜNDACH
TARGET: KUMULIERTE FÖRDERUNG VON PV UND GRÜNDACH
MODALITY: PROPOSAL
DESCRIPTION:
[RELATION_TYPE=SUPPORTED_BY] [MODALITY=PROPOSAL] Green roofs, in combination with PV, can receive cumu

In [26]:
predicate_group_actions["SUPPORTED_BY"] = "REVIEW_REQUIRED"

invalid_predicate_examples["group_action"] = (
    invalid_predicate_examples["relation_type_display"]
    .map(predicate_group_actions)
)

print(
    invalid_predicate_examples[
        ["relation_type_display", "group_action"]
    ]
    .value_counts()
    .sort_index()
)

relation_type_display  group_action            
AMENDS                 QUARANTINE                    1
ASSOCIATED_WITH        REVIEW_REQUIRED             107
BENEFITS_FROM          REVIEW_REQUIRED               1
CAN_USE                REVIEW_REQUIRED               1
IMPLEMENTED            REVIEW_REQUIRED               1
IMPLEMENTED_BY         POSSIBLE_INVERSE_MAPPING      2
MISSING                MISSING_STRUCTURE             9
SUPPORTED_BY           REVIEW_REQUIRED              11
Name: count, dtype: int64


## Testing the `IMPLEMENTED_BY` inverse-mapping hypothesis

`IMPLEMENTED_BY` is not part of Schema V1.1.

A possible normalization is:

`A -- IMPLEMENTED_BY --> B`

→

`B -- IMPLEMENTS --> A`

Before applying this rule automatically, the two Pilot 2 `IMPLEMENTED_BY`
relationships are inspected to verify that reversing source and target preserves
their meaning.

In [27]:
implemented_by_cases = relationships_checked[
    relationships_checked["relation_type"] == "IMPLEMENTED_BY"
].copy()

print("Number of IMPLEMENTED_BY relationships:", len(implemented_by_cases))

for _, row in implemented_by_cases.sort_values("human_readable_id").iterrows():
    print("=" * 100)
    print("ID:", row["human_readable_id"])
    print("SOURCE:", row["source"])
    print("TARGET:", row["target"])
    print("MODALITY:", row["modality"])
    print("DESCRIPTION:")
    print(row["description"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])
    print()

Number of IMPLEMENTED_BY relationships: 2
ID: 253
SOURCE: ZUSCHLÄGE BEI INNOVATIVEN ANLAGENFORMEN
TARGET: ERNEUERBAREN-AUSBAU-GESETZ (EAG)
MODALITY: EXPLICIT_FACT
DESCRIPTION:
[RELATION_TYPE=IMPLEMENTED_BY] [MODALITY=EXPLICIT_FACT] Financial premiums for innovative PV system types are implemented via the Renewable Expansion Act (EAG).
TEXT UNIT IDS:
['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24']

ID: 254
SOURCE: ABSCHLÄGE BEI PV-FREIFLÄCHENANLAGEN
TARGET: ERNEUERBAREN-AUSBAU-GESETZ (EAG)
MODALITY: EXPLICIT_FACT
DESCRIPTION:
[RELATION_TYPE=IMPLEMENTED_BY] [MODALITY=EXPLICIT_FACT] Financial deductions for ground-mounted PV installations are implemented by the Renewable Expansion Act (EAG).
TEXT UNIT IDS:
['23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24']



In [28]:
implemented_by_text_unit_id = (
    implemented_by_cases.iloc[0]["text_unit_ids"][0]
)

implemented_by_evidence = text_units[
    text_units["id"] == implemented_by_text_unit_id
]

print("TEXT UNIT ID:")
print(implemented_by_text_unit_id)

print("\nSOURCE TEXT:\n")
print(implemented_by_evidence.iloc[0]["text"])

TEXT UNIT ID:
23663988970639bea940978ccbd9150fd292856f2f31a1d587392606a64713d5dfeade66f1beea67d1cc0b0e93012337685e0eca3267f98c1695a466657a8b24

SOURCE TEXT:

 bevorzugte PV-An-

wendungen, sowie eine Aufteilung der Fördermittel in Größenklassen sind darin bereits ent-
halten.  Auch  die  Novellierung  des  Wohnungseigentumsgesetzes  im  Jänner  20227 hat  Er-
leichterungen für den Bau von Photovoltaikanlagen bei Reihenhäusern oder Einzelgebäu-

den gebracht. Das Anfang 2024 in Begutachtung befindliche Elektrizitätswirtschaftsgesetz

(ElWG) schafft neue und zeitgemäße Spielregeln für den Strommarkt. Durch mehr Transpa-

renz im Netz, neue Marktrollen und Maßnahmen für mehr Flexibilität wird ein wichtiger

Beitrag zur schnelleren Integration von erneuerbaren Energieanlagen geschaffen. Weiters

wird auch das in Vorbereitung stehende Erneuerbaren-Ausbau-Beschleunigungsgesetz (E-

ABG)  zu  einer  Verfahrensbeschleunigung  bei  der  Errichtung  von  PV-Anlagen  und  für  die

Energiewende n

### Evaluation of `IMPLEMENTED_BY`

The supporting source text states:

> "Mit dem EAG wurden entsprechende Anreize in Form von Zuschlägen bei innovativen Anlagenformen sowie Abschlägen bei PV-Freiflächenanlagen gesetzt."

This directly supports the interpretation that the Renewable Expansion Act (EAG)
implements or establishes the two support measures.

Therefore, for the two Pilot 2 `IMPLEMENTED_BY` relationships, the inverse
normalization is semantically valid:

`A -- IMPLEMENTED_BY --> B`

becomes:

`B -- IMPLEMENTS --> A`

The modality remains `EXPLICIT_FACT`.

**Pilot 2 normalization rule**

`IMPLEMENTED_BY` → reverse source/target + replace predicate with `IMPLEMENTS`

This rule is accepted for the two inspected Pilot 2 cases. It should not be
assumed to be universally valid for future datasets without validation.

In [29]:
predicate_group_actions["IMPLEMENTED_BY"] = "AUTO_NORMALIZE_INVERSE"

invalid_predicate_examples["group_action"] = (
    invalid_predicate_examples["relation_type_display"]
    .map(predicate_group_actions)
)

print(
    invalid_predicate_examples[
        ["relation_type_display", "group_action"]
    ]
    .value_counts()
    .sort_index()
)

relation_type_display  group_action          
AMENDS                 QUARANTINE                  1
ASSOCIATED_WITH        REVIEW_REQUIRED           107
BENEFITS_FROM          REVIEW_REQUIRED             1
CAN_USE                REVIEW_REQUIRED             1
IMPLEMENTED            REVIEW_REQUIRED             1
IMPLEMENTED_BY         AUTO_NORMALIZE_INVERSE      2
MISSING                MISSING_STRUCTURE           9
SUPPORTED_BY           REVIEW_REQUIRED            11
Name: count, dtype: int64


## Canonical endpoint mapping

Relationship normalization requires canonical source and target entities.

The current canonical entity table contains only the entities already validated
during the normalization prototype.

Each raw relationship source and target is therefore compared with:

- canonical labels
- known aliases

If a match is found, the raw endpoint is mapped to its canonical ID.

An unmapped endpoint is **not treated as incorrect**. It means that entity
resolution for that endpoint has not yet been completed.

In [30]:
def normalize_entity_label(value):
    if pd.isna(value):
        return None

    return " ".join(
        str(value).strip().upper().split()
    )


# Build alias/canonical-label lookup
canonical_lookup = {}

for _, row in canonical_entities.iterrows():

    labels = [row["canonical_label"]]

    if pd.notna(row["aliases"]):
        labels.extend(
            [
                alias.strip()
                for alias in str(row["aliases"]).split(";")
                if alias.strip()
            ]
        )

    for label in labels:
        canonical_lookup[
            normalize_entity_label(label)
        ] = row["canonical_id"]


# Map relationship endpoints
relationships_checked["canonical_source_id"] = (
    relationships_checked["source"]
    .apply(
        lambda x: canonical_lookup.get(
            normalize_entity_label(x)
        )
    )
)

relationships_checked["canonical_target_id"] = (
    relationships_checked["target"]
    .apply(
        lambda x: canonical_lookup.get(
            normalize_entity_label(x)
        )
    )
)

relationships_checked["both_endpoints_mapped"] = (
    relationships_checked["canonical_source_id"].notna()
    &
    relationships_checked["canonical_target_id"].notna()
)


print("Total relationships:", len(relationships_checked))

print(
    "Source endpoint mapped:",
    relationships_checked["canonical_source_id"].notna().sum()
)

print(
    "Target endpoint mapped:",
    relationships_checked["canonical_target_id"].notna().sum()
)

print(
    "Both endpoints mapped:",
    relationships_checked["both_endpoints_mapped"].sum()
)

Total relationships: 643
Source endpoint mapped: 62
Target endpoint mapped: 83
Both endpoints mapped: 5


## Relationships with both canonical endpoints available

Only relationships whose source and target have both already been resolved to
canonical entities can currently be considered fully ready for controlled-KG
normalization.

Because the canonical entity table is still a small prototype, only a limited
number of Pilot 2 relationships are expected to satisfy this condition.

The following relationships are inspected to determine whether they are already
represented in the controlled KG or still require validation.

In [31]:
mapped_relationships = relationships_checked[
    relationships_checked["both_endpoints_mapped"]
].copy()

print(
    "Relationships with both endpoints mapped:",
    len(mapped_relationships)
)

mapped_relationships[
    [
        "human_readable_id",
        "source",
        "relation_type",
        "target",
        "modality",
        "canonical_source_id",
        "canonical_target_id",
        "validation_flag",
        "description"
    ]
]

Relationships with both endpoints mapped: 5


,human_readable_id,source,relation_type,target,modality,canonical_source_id,canonical_target_id,validation_flag,description
0,0,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",RESPONSIBLE_FOR,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,EXPLICIT_FACT,ORG_0001,POL_0001,STRUCTURALLY_OK,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=EXPL...
257,257,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,EXPLICIT_FACT,TEC_0002,TEC_0001,STRUCTURALLY_OK,[RELATION_TYPE=PART_OF] [MODALITY=EXPLICIT_FAC...
425,425,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,AGRI-PV-ANLAGE,EXPLICIT_FACT,POL_0001,TEC_0002,STRUCTURALLY_OK,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...
543,543,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,PV-ANLAGEN,EXPLICIT_FACT,POL_0001,TEC_0001,STRUCTURALLY_OK,[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FA...
642,642,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",RESPONSIBLE_FOR,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,REASONABLE_INFERENCE,ORG_0001,POL_0001,STRUCTURALLY_OK,[RELATION_TYPE=RESPONSIBLE_FOR] [MODALITY=REAS...


## Case 4 — Remaining fully mapped relationship

Relationship 543 is the only fully mapped relationship not yet evaluated.

Raw relationship:

`ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE -- SUPPORTS --> PV-ANLAGEN`

Canonical endpoints:

`POL_0001 -- SUPPORTS --> TEC_0001`

The relationship is structurally compliant with Schema V1.1, but semantic
support and modality must still be checked against the source evidence.

In [32]:
relation_543 = relationships[
    relationships["human_readable_id"] == 543
]

for _, row in relation_543.iterrows():
    print("GRAPH RELATIONSHIP ID:", row["id"])
    print("SOURCE:", row["source"])
    print("TARGET:", row["target"])
    print("DESCRIPTION:")
    print(row["description"])
    print("WEIGHT:", row["weight"])
    print("TEXT UNIT IDS:")
    print(row["text_unit_ids"])

GRAPH RELATIONSHIP ID: b7feebe2-c163-4987-a25f-b9462ce9de85
SOURCE: ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
TARGET: PV-ANLAGEN
DESCRIPTION:
[RELATION_TYPE=SUPPORTS] [MODALITY=EXPLICIT_FACT] The Austrian Photovoltaic Strategy aims to support the deployment and advancement of PV installations.
WEIGHT: 10.0
TEXT UNIT IDS:
['dafd7f2f39e49c87969ec1360e9fae0af9de6f290655205c8c120dd24dcc85da04f0536237f7f9529d31deed892d5abae3b8f58fb66ae9e497327f23b1e6b750']


In [33]:
text_unit_543_id = relation_543.iloc[0]["text_unit_ids"][0]

evidence_543 = text_units[
    text_units["id"] == text_unit_543_id
]

print("TEXT UNIT ID:")
print(text_unit_543_id)

print("\nSOURCE TEXT:\n")
print(evidence_543.iloc[0]["text"])

TEXT UNIT ID:
dafd7f2f39e49c87969ec1360e9fae0af9de6f290655205c8c120dd24dcc85da04f0536237f7f9529d31deed892d5abae3b8f58fb66ae9e497327f23b1e6b750

SOURCE TEXT:

iewende“ hat das

Ziel, Lösungen zur Erreichung der Klimaneutralität 2040 zu entwickeln und zu demonstrie-

ren. Dies geschieht durch die Erforschung und Entwicklung klimafreundlicher Energietech-

nologien und -komponenten „Made in Austria“. Dadurch werden die Technologiekompe-

tenzen  sowie  der  Innovationsstandort  Österreich  bei  gleichzeitiger  Verbesserung  der  Ex-

portchancen gestärkt. Der Umsetzungsplan des FTI-Schwerpunkts „Energiewende“ konkre-

tisiert hierbei auch die Ziele des Nationalen Energie- und Klimaplans durch die gesetzten

Innovationsziele.

32 von 44

Österreichische Photovoltaik-Strategie

6.6.1.2  PV-Kreislaufwirtschaft & Produktion
Österreichische Unternehmen sollen – unterstützt durch nationale und transnationale FTI-

Aktivitäten – in die Lage gebracht werden, an den solaren Wertschöpfungsketten d

### Evaluation

The supporting source text contains policy-oriented statements concerning
funding and incentives for PV installations, including:

> "Bei Förderungen und Anreizen für PV-Anlagen ist darauf zu achten ..."

and statements that particular incentives are considered appropriate.

This supports the controlled relationship:

`POL_0001 -- SUPPORTS --> TEC_0001`

However, the GraphRAG modality `EXPLICIT_FACT` does not adequately represent
the normative character of the source. The strategy recommends and promotes
measures supporting PV deployment rather than merely stating a neutral fact.

**Decision: CORRECTED**

Normalized relationship:

`REL_0003: POL_0001 -- SUPPORTS --> TEC_0001`

- Predicate: `SUPPORTS`
- Modality: `RECOMMENDATION`
- Validation status: `CORRECTED`

The relationship is retained in the controlled KG, but its modality is corrected
from `EXPLICIT_FACT` to `RECOMMENDATION`.

In [34]:
third_normalized_relationship = pd.DataFrame([
    {
        "relationship_id": "REL_0003",
        "source_id": "POL_0001",
        "predicate": "SUPPORTS",
        "target_id": "TEC_0001",
        "modality": "RECOMMENDATION",
        "status": "",
        "raw_relationship_ids": relation_543.iloc[0]["id"],
        "text_unit_ids": text_unit_543_id,
        "validation_status": "CORRECTED",
        "normalization_note": (
            "The source contains policy recommendations and incentives supporting "
            "PV installations. SUPPORTS is retained, but GraphRAG modality "
            "EXPLICIT_FACT is corrected to RECOMMENDATION."
        )
    }
])

third_normalized_relationship

,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0003,POL_0001,SUPPORTS,TEC_0001,RECOMMENDATION,,b7feebe2-c163-4987-a25f-b9462ce9de85,dafd7f2f39e49c87969ec1360e9fae0af9de6f29065520...,CORRECTED,The source contains policy recommendations and...


In [35]:
normalized_relationships = pd.read_csv(normalized_relationships_path)

if "REL_0003" not in normalized_relationships["relationship_id"].values:
    normalized_relationships = pd.concat(
        [normalized_relationships, third_normalized_relationship],
        ignore_index=True
    )

    normalized_relationships.to_csv(
        normalized_relationships_path,
        index=False
    )

    print("REL_0003 saved successfully.")
else:
    print("REL_0003 already exists.")

normalized_relationships

REL_0003 saved successfully.


,relationship_id,source_id,predicate,target_id,modality,status,raw_relationship_ids,text_unit_ids,validation_status,normalization_note
0,REL_0001,ORG_0001,RESPONSIBLE_FOR,POL_0001,EXPLICIT_FACT,NaN,766acd95-2d5a-4286-bf19-3455bb2616ec; 0d51bb6b...,d7219f76b31818b7b13bf08dda81003d752e53f7660831...,CORRECTED,Collapsed two duplicate GraphRAG relationships...
1,REL_0002,POL_0001,SUPPORTS,TEC_0002,RECOMMENDATION,NaN,94f1ba26-130a-4a14-9023-f55ecf185196,4d221520fe7090409cfa9cda598db30683356c340ab358...,CORRECTED,The source explicitly recommends continued pro...
2,REL_0003,POL_0001,SUPPORTS,TEC_0001,RECOMMENDATION,,b7feebe2-c163-4987-a25f-b9462ce9de85,dafd7f2f39e49c87969ec1360e9fae0af9de6f29065520...,CORRECTED,The source contains policy recommendations and...


In [36]:
relation_543_decision = pd.DataFrame([
    {
        "raw_relationship_id": relation_543.iloc[0]["id"],
        "human_readable_id": 543,
        "raw_source": "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE",
        "raw_predicate": "SUPPORTS",
        "raw_target": "PV-ANLAGEN",
        "canonical_source_id": "POL_0001",
        "canonical_target_id": "TEC_0001",
        "decision": "CORRECTED",
        "reason": (
            "The predicate SUPPORTS is supported by the source, but the raw modality "
            "EXPLICIT_FACT does not reflect the normative policy wording. "
            "The normalized modality is RECOMMENDATION."
        ),
        "text_unit_id": text_unit_543_id
    }
])

relation_543_decision

,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,b7feebe2-c163-4987-a25f-b9462ce9de85,543,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,PV-ANLAGEN,POL_0001,TEC_0001,CORRECTED,The predicate SUPPORTS is supported by the sou...,dafd7f2f39e49c87969ec1360e9fae0af9de6f29065520...


In [37]:
relationship_decisions = pd.read_csv(relationship_decisions_path)

if 543 not in relationship_decisions["human_readable_id"].values:
    relationship_decisions = pd.concat(
        [relationship_decisions, relation_543_decision],
        ignore_index=True
    )

    relationship_decisions.to_csv(
        relationship_decisions_path,
        index=False
    )

    print("Relationship 543 decision saved.")
else:
    print("Relationship 543 decision already exists.")

relationship_decisions

Relationship 543 decision saved.


,raw_relationship_id,human_readable_id,raw_source,raw_predicate,raw_target,canonical_source_id,canonical_target_id,decision,reason,text_unit_id
0,1b0a059f-694c-4e24-a3c7-766ebedc83a4,257,AGRI-PV-ANLAGE,PART_OF,FOTOVOLTAIKANLAGE,TEC_0002,TEC_0001,REJECTED,The source discusses Agri-PV as a form/applica...,23663988970639bea940978ccbd9150fd292856f2f31a1...
1,94f1ba26-130a-4a14-9023-f55ecf185196,425,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,AGRI-PV-ANLAGE,POL_0001,TEC_0002,CORRECTED,The predicate SUPPORTS is supported by the sou...,4d221520fe7090409cfa9cda598db30683356c340ab358...
2,b7feebe2-c163-4987-a25f-b9462ce9de85,543,ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE,SUPPORTS,PV-ANLAGEN,POL_0001,TEC_0001,CORRECTED,The predicate SUPPORTS is supported by the sou...,dafd7f2f39e49c87969ec1360e9fae0af9de6f29065520...


In [38]:
relationships_checked["relation_type_display"] = (
    relationships_checked["relation_type"]
    .fillna("MISSING")
)

relationships_checked["group_action"] = (
    relationships_checked["relation_type_display"]
    .map(predicate_group_actions)
    .fillna("SCHEMA_COMPLIANT")
)

relationship_validation_path = (
    controlled_kg_path / "relationship_validation_working.csv"
)

relationships_checked.to_csv(
    relationship_validation_path,
    index=False
)

print("Saved:", relationship_validation_path)

print("\nStructural validation:")
print(
    relationships_checked["validation_flag"]
    .value_counts()
)

print("\nPredicate-group treatment:")
print(
    relationships_checked["group_action"]
    .value_counts()
)

Saved: ..\..\kg\controlled_kg\relationship_validation_working.csv

Structural validation:
validation_flag
STRUCTURALLY_OK                   510
INVALID_PREDICATE                 124
INVALID_PREDICATE_AND_MODALITY      9
Name: count, dtype: int64

Predicate-group treatment:
group_action
SCHEMA_COMPLIANT          510
REVIEW_REQUIRED           121
MISSING_STRUCTURE           9
AUTO_NORMALIZE_INVERSE      2
QUARANTINE                  1
Name: count, dtype: int64


# Relationship Normalization — Prototype Summary

## Scope

This notebook developed and evaluated a controlled relationship-normalization
procedure for the 643 raw relationships extracted by GraphRAG in Pilot 2.

The purpose was not to manually validate every relationship, but to identify
representative error patterns and build a repeatable validation workflow.

## Manual validation cases

### Case 1 — Rejected relationship

Raw relationship:

`AGRI-PV-ANLAGE -- PART_OF --> FOTOVOLTAIKANLAGE`

After tracing the relationship to its supporting source text, the predicate
`PART_OF` was found to overstate the evidence.

Decision:

**REJECTED**

---

### Case 2 — Duplicate relationships

Two GraphRAG relationships represented the same underlying relationship between
the BMK and the Austrian Photovoltaic Strategy.

They were collapsed into:

`REL_0001: ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

Modality:

`EXPLICIT_FACT`

The raw relationship IDs and source text-unit IDs were preserved as provenance.

---

### Case 3 — Modality correction

Raw relationship:

`ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE -- SUPPORTS --> AGRI-PV-ANLAGE`

GraphRAG modality:

`EXPLICIT_FACT`

Source evidence showed normative language recommending continued promotion of
Agri-PV.

Normalized relationship:

`REL_0002: POL_0001 -- SUPPORTS --> TEC_0002`

Normalized modality:

`RECOMMENDATION`

Decision:

**CORRECTED**

---

### Case 4 — Modality correction

Raw relationship:

`ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE -- SUPPORTS --> PV-ANLAGEN`

The relationship itself was supported, but the source contained policy-oriented
and normative language rather than a neutral factual statement.

Normalized relationship:

`REL_0003: POL_0001 -- SUPPORTS --> TEC_0001`

Normalized modality:

`RECOMMENDATION`

Decision:

**CORRECTED**

---

## Automatic structural validation

The GraphRAG relationship descriptions were parsed to extract:

- `RELATION_TYPE`
- `MODALITY`

These values were checked against Schema V1.1.

Results:

- Total raw relationships: **643**
- Structurally compliant: **510**
- Invalid predicate only: **124**
- Invalid predicate and modality: **9**
- Invalid modality only: **0**

Passing structural validation does not imply that a relationship is semantically
correct.

For example, relationship 257 used valid Schema V1.1 values but was rejected
after source-evidence inspection.

---

## Out-of-schema predicate treatment

The structurally problematic relationships were grouped by predicate.

Final treatment categories:

- `SCHEMA_COMPLIANT`: **510**
- `REVIEW_REQUIRED`: **121**
- `MISSING_STRUCTURE`: **9**
- `AUTO_NORMALIZE_INVERSE`: **2**
- `QUARANTINE`: **1**

### Review required

Predicates such as:

- `ASSOCIATED_WITH`
- `SUPPORTED_BY`
- `BENEFITS_FROM`
- `CAN_USE`
- `IMPLEMENTED`

were not automatically mapped because their semantics were heterogeneous or
too vague.

### Missing structure

Nine relationships did not contain a usable controlled predicate/modality
structure and require later review.

### Quarantine

The predicate:

`AMENDS`

was generated once by GraphRAG but is not part of Schema V1.1.

It is therefore quarantined rather than automatically added to the schema.

### Validated automatic normalization rule

Two relationships used:

`A -- IMPLEMENTED_BY --> B`

Source verification confirmed that for these Pilot 2 cases they can be normalized as:

`B -- IMPLEMENTS --> A`

Therefore:

`IMPLEMENTED_BY`

is handled as:

**reverse endpoints + replace predicate with `IMPLEMENTS`**

for these verified Pilot 2 cases.

This rule should not automatically be assumed valid for future datasets without
additional validation.

---

## Canonical endpoint mapping

Raw relationship endpoints were compared with the canonical labels and aliases
currently present in the controlled entity table.

Results:

- source endpoints mapped: **62**
- target endpoints mapped: **83**
- both endpoints mapped: **5**

Only four canonical entities have currently been created, so the low endpoint
coverage is expected and does not indicate extraction failure.

All five relationships whose two endpoints were already canonicalized have now
been accounted for:

- raw relationships `0` and `642` → `REL_0001`
- raw relationship `257` → rejected
- raw relationship `425` → `REL_0002`
- raw relationship `543` → `REL_0003`

---

## Current controlled relationships

The prototype controlled KG currently contains:

`REL_0001: ORG_0001 -- RESPONSIBLE_FOR --> POL_0001`

`REL_0002: POL_0001 -- SUPPORTS --> TEC_0002`

`REL_0003: POL_0001 -- SUPPORTS --> TEC_0001`

---

## Output files

The relationship-normalization stage produces:

- `normalized_relationships_working.csv`
  - relationships retained in the controlled KG

- `relationship_decisions_working.csv`
  - manually reviewed relationship decisions

- `relationship_validation_working.csv`
  - automatic structural validation results for all 643 raw relationships

---

## Conclusion

GraphRAG provides useful relationship candidates but should not be treated as
an authoritative Knowledge Graph.

The Pilot 2 experiment demonstrated several error patterns:

- duplicate relationships
- incorrect predicates
- incorrect modality
- vague generic predicates
- missing structured relation information
- out-of-schema predicates
- inconsistent relationship direction

A controlled normalization layer can detect many structural problems
automatically while reserving source-evidence review for selected ambiguous or
high-risk cases.

The relationship-normalization prototype is therefore considered complete.

The next project stage should build on this controlled subset rather than
attempting to manually validate all 643 GraphRAG relationships.